# 10 — Aggregation, Ensemble, Ablations

This is the final notebook of the benchmark. It does **not** train any
model. It reads everything the previous notebooks left in
`MyDrive/melanoma/results/` and `MyDrive/melanoma/checkpoints/` and
produces every table and figure the IEEE paper needs.

Run on a **CPU runtime** to save GPU compute units. The only GPU step is
the ensemble's val/test inference for any CNN whose `*_val_predictions.csv`
is missing — set the runtime to **A100 / L4** if you want that step to
run in seconds rather than minutes. (It can also run on CPU; just slower.)

In [ ]:
# --- Colab setup: ensure the project is on sys.path, mount Drive, load config ---
import os, sys, subprocess
from pathlib import Path

# Either the project is already on disk (uploaded zip / mounted Drive) or we
# clone it from GitHub. We never destroy local changes.
REPO_URL = "https://github.com/zkoymen/melanoma-detection-ham10000.git"
CANDIDATE_PATHS = [
    Path.cwd(),
    Path("/content/melanoma-detection-ham10000"),
    Path("/content/drive/MyDrive/melanoma-detection-ham10000"),
]

project_root = None
for p in CANDIDATE_PATHS:
    if (p / "src").exists() and (p / "config.py").exists():
        project_root = p
        break

if project_root is None:
    project_root = Path("/content/melanoma-detection-ham10000")
    subprocess.run(["git", "clone", REPO_URL, str(project_root)], check=True)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Mount Drive (silently re-uses an existing mount on re-run)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except (ImportError, ModuleNotFoundError):
    pass

import config
config.ensure_drive_dirs()
print("Project root:", project_root)
print("Drive root  :", config.DRIVE_ROOT)
print("Data dir    :", config.DATA_DIR)
print("Results dir :", config.RESULTS_DIR)

In [ ]:
import random, numpy as np, torch
random.seed(config.SEED); np.random.seed(config.SEED); torch.manual_seed(config.SEED)
torch.cuda.manual_seed_all(config.SEED)

In [ ]:
!pip install --quiet timm 2>&1 | tail -n 1

In [ ]:
# --- Inputs we expect ---
import json, numpy as np, pandas as pd, torch
from pathlib import Path

ARCHES = ["alexnet", "vgg16_bn", "resnet50", "efficientnet_b3", "densenet121", "swin_tiny"]
NON_DEEP = ["baseline_logistic", "classical_ml_svm"]
LEGACY_FOR_ABLATION = "deep_learning_effnet"  # old EfficientNet-B0, softened CW + light aug

print("Listing results dir:")
for p in sorted(config.RESULTS_DIR.iterdir()):
    print(" ", p.name)

In [ ]:
# --- Build a working DataFrame of every method's headline metrics ---
def load_metrics(name):
    p = config.RESULTS_DIR / f"{name}_metrics.json"
    if not p.exists():
        return None
    return json.loads(p.read_text())

rows = []
for name in NON_DEEP + ARCHES:
    m = load_metrics(name)
    if m is None:
        print(f"  WARN: missing {name}_metrics.json")
        continue
    rows.append({
        "method":    name,
        "accuracy":  m.get("accuracy"),
        "precision": m.get("precision"),
        "recall":    m.get("recall"),
        "f1":        m.get("f1"),
        "roc_auc":   m.get("roc_auc"),
        "train_time_sec": m.get("train_time_sec"),
        "inference_ms_per_img": m.get("inference_time_per_image_ms"),
    })
df_pre = pd.DataFrame(rows)

# Method 10 (hybrid fusion) is optional — append if its metrics file exists.
hf = load_metrics("hybrid_fusion")
if hf is not None:
    df_pre = pd.concat([df_pre, pd.DataFrame([{
        "method":    "hybrid_fusion",
        "accuracy":  hf.get("accuracy"),
        "precision": hf.get("precision"),
        "recall":    hf.get("recall"),
        "f1":        hf.get("f1"),
        "roc_auc":   hf.get("roc_auc"),
        "train_time_sec": hf.get("train_time_sec"),
        "inference_ms_per_img": hf.get("inference_time_per_image_ms"),
    }])], ignore_index=True)
    print("  + hybrid_fusion row included.")

print(df_pre.to_string(index=False))

In [ ]:
# --- Compute the soft-vote ensemble (method 9) ---
# We need val + test probabilities from each CNN. Test probs are in
# {arch}_predictions.csv (column y_prob_tta). Val probs are in
# {arch}_val_predictions.csv. If a val CSV is missing (e.g. ResNet50 from
# Phase 1), reload that arch's checkpoint and recompute val + test predictions.

import torch
from torch.utils.data import DataLoader
from src.data import load_arrays, HAMDataset, make_eval_transform
from src.models import BUILDERS
from src.training import predict, tta_predict, tune_threshold

X, y, ids, idx_train, idx_val, idx_test = load_arrays(config.DATA_DIR)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

def recompute_val_predictions(arch):
    """Reload checkpoint and re-do val + test passes; save both CSVs."""
    print(f"  recomputing val/test predictions for {arch}...")
    cfg = config.ARCH_CONFIG[arch]
    model = BUILDERS[arch](num_classes=2, pretrained=False).to(device)
    ckpt = torch.load(config.CHECKPOINT_DIR / f"{arch}_best.pt", map_location=device)
    model.load_state_dict(ckpt)
    model.eval()
    eval_tf = make_eval_transform(cfg["input_size"])
    val_ld  = DataLoader(HAMDataset(X, y, idx_val,  eval_tf), batch_size=cfg["batch_size"],
                         shuffle=False, num_workers=2, pin_memory=True)
    test_ld = DataLoader(HAMDataset(X, y, idx_test, eval_tf), batch_size=cfg["batch_size"],
                         shuffle=False, num_workers=2, pin_memory=True)

    yv, _, pv = predict(model, val_ld, device)
    yv2, pv_tta = tta_predict(model, val_ld, device, config.TTA_TRANSFORMS)
    yt, _, pt = predict(model, test_ld, device)
    yt2, pt_tta = tta_predict(model, test_ld, device, config.TTA_TRANSFORMS)

    pd.DataFrame({"image_id": ids[idx_val], "y_true": yv,
                  "y_prob_single": pv, "y_prob_tta": pv_tta}
                ).to_csv(config.RESULTS_DIR / f"{arch}_val_predictions.csv", index=False)

    # Refresh the test CSV too — but only if it doesn't already have y_prob_tta
    test_csv = config.RESULTS_DIR / f"{arch}_predictions.csv"
    if test_csv.exists():
        existing = pd.read_csv(test_csv)
        if "y_prob_tta" in existing.columns:
            return  # already rich; don't overwrite
    bt, _ = tune_threshold(yv, pv)
    bt_tta, _ = tune_threshold(yv2, pv_tta)
    pd.DataFrame({"image_id": ids[idx_test], "y_true": yt,
                  "y_pred_single": (pt > bt).astype(int),     "y_prob_single": pt,
                  "y_pred_tta":    (pt_tta > bt_tta).astype(int), "y_prob_tta":    pt_tta,
                  "best_t_single": bt, "best_t_tta": bt_tta,
                 }).to_csv(test_csv, index=False)

# Verify each CNN has val_predictions.csv; recompute from checkpoint if missing.
# CNNs whose checkpoint ALSO doesn't exist are dropped from ARCHES so the
# ensemble row and downstream tables build on whatever is actually trained.
TRAINED_ARCHES = []
for arch in ARCHES:
    val_csv = config.RESULTS_DIR / f"{arch}_val_predictions.csv"
    ckpt    = config.CHECKPOINT_DIR / f"{arch}_best.pt"
    test_csv = config.RESULTS_DIR / f"{arch}_predictions.csv"
    if val_csv.exists() and test_csv.exists():
        TRAINED_ARCHES.append(arch)
        print(f"  ok: {arch}_val_predictions.csv present")
    elif ckpt.exists() and test_csv.exists():
        recompute_val_predictions(arch)
        TRAINED_ARCHES.append(arch)
    else:
        print(f"  SKIP: {arch} (no checkpoint and no predictions on Drive)")

if not TRAINED_ARCHES:
    raise RuntimeError("No CNN checkpoints or predictions found on Drive. "
                       "Run at least one of notebooks 04-09 before this aggregation.")
print(f"\nEnsemble will average over {len(TRAINED_ARCHES)} CNN(s): {TRAINED_ARCHES}")
ARCHES = TRAINED_ARCHES   # rebind so downstream cells use only trained models

In [ ]:
# --- Build the ensemble: average TTA probabilities across the 6 CNNs ---
val_probs_list, test_probs_list = [], []
y_val_true_ref, y_test_true_ref = None, None

for arch in ARCHES:
    vdf = pd.read_csv(config.RESULTS_DIR / f"{arch}_val_predictions.csv")
    tdf = pd.read_csv(config.RESULTS_DIR / f"{arch}_predictions.csv")
    val_probs_list.append(vdf["y_prob_tta"].to_numpy())
    test_probs_list.append(tdf["y_prob_tta"].to_numpy())
    if y_val_true_ref  is None: y_val_true_ref  = vdf["y_true"].to_numpy()
    if y_test_true_ref is None: y_test_true_ref = tdf["y_true"].to_numpy()

ens_val_prob  = np.mean(val_probs_list,  axis=0)
ens_test_prob = np.mean(test_probs_list, axis=0)

best_t_ens, val_f1_ens = tune_threshold(y_val_true_ref, ens_val_prob)
print(f"Ensemble val F1 = {val_f1_ens:.4f}  at threshold = {best_t_ens:.3f}")

ens_test_pred = (ens_test_prob > best_t_ens).astype(int)

In [ ]:
# --- Save ensemble outputs in the standard layout ---
from src.evaluation import save_standard_outputs, compute_metrics

ens_metrics = save_standard_outputs(
    method_name="ensemble",
    results_dir=config.RESULTS_DIR,
    y_true=y_test_true_ref,
    y_pred=ens_test_pred,
    y_prob=ens_test_prob,
    ids=pd.read_csv(config.RESULTS_DIR / f"{ARCHES[0]}_predictions.csv")["image_id"].to_numpy(),
    hyperparameters={
        "members": ARCHES,
        "weights": "uniform soft-vote",
        "decision_threshold_tta": float(best_t_ens),
    },
    train_time_sec=0.0,
    inference_time_per_image_ms=float(np.mean(
        [load_metrics(a).get("inference_time_per_image_ms", 0.0) for a in ARCHES])),
)
print({k: round(v, 4) for k, v in ens_metrics.items() if isinstance(v, (int, float))})

# Append to the dataframe
df_pre = pd.concat([df_pre, pd.DataFrame([{
    "method":   "ensemble",
    "accuracy": ens_metrics["accuracy"],
    "precision":ens_metrics["precision"],
    "recall":   ens_metrics["recall"],
    "f1":       ens_metrics["f1"],
    "roc_auc":  ens_metrics["roc_auc"],
    "train_time_sec": 0.0,
    "inference_ms_per_img": ens_metrics["inference_time_per_image_ms"],
}])], ignore_index=True)

In [ ]:
# --- Save the headline 9-row comparison table ---
df_pre = df_pre.round(4)
df_pre.to_csv(config.RESULTS_DIR / "comparison_table.csv", index=False)
print(df_pre.to_string(index=False))

In [ ]:
# --- Build the val-F1 vs epoch overlay (figure required by the syllabus) ---
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
for arch in ARCHES:
    npz = config.RESULTS_DIR / f"{arch}_curves.npz"
    if not npz.exists():
        print(f"  missing {npz.name}")
        continue
    d = np.load(npz)
    ax.plot(np.arange(1, len(d["val_f1"]) + 1), d["val_f1"], label=arch)
ax.set_xlabel("Epoch (stage-2 fine-tuning)")
ax.set_ylabel("Validation F1")
ax.set_title("Per-epoch validation F1 — 6 CNN architectures")
ax.legend(loc="lower right", fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(config.RESULTS_DIR / "epoch_curves.png", dpi=120)
plt.show()

In [ ]:
# --- Build the 9-curve ROC overlay ---
from sklearn.metrics import roc_curve, roc_auc_score
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="chance")
for name in NON_DEEP + ARCHES + ["ensemble", "hybrid_fusion"]:
    csv = config.RESULTS_DIR / f"{name}_predictions.csv"
    if not csv.exists():
        continue
    pdf = pd.read_csv(csv)
    if "y_prob_tta" in pdf.columns:
        prob = pdf["y_prob_tta"].to_numpy()
    elif "y_prob" in pdf.columns:
        prob = pdf["y_prob"].to_numpy()
    else:
        continue
    fpr, tpr, _ = roc_curve(pdf["y_true"], prob)
    auc = roc_auc_score(pdf["y_true"], prob)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC — 9-method comparison on HAM10000 binary test set")
ax.legend(loc="lower right", fontsize=7)
fig.tight_layout()
fig.savefig(config.RESULTS_DIR / "roc_overlay.png", dpi=120)
plt.show()

In [ ]:
# --- Error analysis: 4 FP + 4 FN from the best single CNN ---
best_arch = df_pre[df_pre["method"].isin(ARCHES)].sort_values("f1", ascending=False).iloc[0]["method"]
print("Best single CNN:", best_arch)
pred_csv = config.RESULTS_DIR / f"{best_arch}_predictions.csv"
pdf = pd.read_csv(pred_csv)
fp_rows = pdf[(pdf["y_pred_tta"] == 1) & (pdf["y_true"] == 0)].sort_values("y_prob_tta", ascending=False).head(4)
fn_rows = pdf[(pdf["y_pred_tta"] == 0) & (pdf["y_true"] == 1)].sort_values("y_prob_tta", ascending=True ).head(4)

id_to_arr_idx = {str(s): i for i, s in enumerate(ids)}
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, r) in zip(axes[0], fp_rows.iterrows()):
    k = id_to_arr_idx[str(r["image_id"])]
    ax.imshow(X[k]); ax.set_title(f"FP  p={r['y_prob_tta']:.2f}", fontsize=9); ax.axis("off")
for ax, (_, r) in zip(axes[1], fn_rows.iterrows()):
    k = id_to_arr_idx[str(r["image_id"])]
    ax.imshow(X[k]); ax.set_title(f"FN  p={r['y_prob_tta']:.2f}", fontsize=9); ax.axis("off")
fig.suptitle(f"Error analysis on test set — {best_arch} (top row: false positives, bottom: false negatives)")
fig.tight_layout()
fig.savefig(config.RESULTS_DIR / "error_analysis.png", dpi=120)
plt.show()

In [ ]:
# --- Ablation table (4 rows; no extra training) ---
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, roc_auc_score

def metrics_at_threshold(y_true, y_prob, t):
    y_pred = (y_prob > t).astype(int)
    return dict(
        accuracy=accuracy_score(y_true, y_pred),
        precision=precision_score(y_true, y_pred, zero_division=0),
        recall=recall_score(y_true, y_pred, zero_division=0),
        f1=f1_score(y_true, y_pred, zero_division=0),
        roc_auc=roc_auc_score(y_true, y_prob),
    )

ab_rows = []

# A) Loss/sampler: legacy softened-CW EfficientNet-B0 vs new focal+sampler EfficientNet-B3
m_legacy = load_metrics(LEGACY_FOR_ABLATION)
m_b3     = load_metrics("efficientnet_b3")
if m_legacy and m_b3:
    ab_rows.append({"ablation": "Loss + sampling: softened class-weight (legacy B0)",
                    **{k: m_legacy.get(k) for k in ["accuracy","precision","recall","f1","roc_auc"]}})
    ab_rows.append({"ablation": "Loss + sampling: Focal + WeightedRandomSampler (B3)",
                    **{k: m_b3.get(k) for k in ["accuracy","precision","recall","f1","roc_auc"]}})

# B) TTA on/off — averaged across the 6 CNNs
single_f1s, tta_f1s = [], []
for arch in ARCHES:
    pdf = pd.read_csv(config.RESULTS_DIR / f"{arch}_predictions.csv")
    single_f1s.append(f1_score(pdf["y_true"], pdf["y_pred_single"], zero_division=0))
    tta_f1s   .append(f1_score(pdf["y_true"], pdf["y_pred_tta"],    zero_division=0))
ab_rows.append({"ablation": "TTA OFF (mean across 6 CNNs)", "f1": float(np.mean(single_f1s))})
ab_rows.append({"ablation": "TTA ON  (mean across 6 CNNs)", "f1": float(np.mean(tta_f1s))})

# C) Threshold tuned vs t=0.5 — averaged across the 6 CNNs (TTA probs)
def_f1s, tuned_f1s = [], []
for arch in ARCHES:
    pdf = pd.read_csv(config.RESULTS_DIR / f"{arch}_predictions.csv")
    def_f1s.append(f1_score(pdf["y_true"], (pdf["y_prob_tta"] > 0.5).astype(int), zero_division=0))
    tuned_f1s.append(f1_score(pdf["y_true"], pdf["y_pred_tta"], zero_division=0))
ab_rows.append({"ablation": "Threshold = 0.5 (mean across 6 CNNs)",         "f1": float(np.mean(def_f1s))})
ab_rows.append({"ablation": "Threshold tuned on val (mean across 6 CNNs)",  "f1": float(np.mean(tuned_f1s))})

# D) Best single CNN vs ensemble
best_single = df_pre[df_pre["method"].isin(ARCHES)]["f1"].max()
ens_f1      = df_pre[df_pre["method"] == "ensemble"]["f1"].iloc[0]
ab_rows.append({"ablation": "Best single CNN (test F1)", "f1": float(best_single)})
ab_rows.append({"ablation": "Ensemble (soft-vote, 6 CNNs)", "f1": float(ens_f1)})

ab_df = pd.DataFrame(ab_rows).round(4)
ab_df.to_csv(config.RESULTS_DIR / "ablation_table.csv", index=False)
print(ab_df.to_string(index=False))

In [ ]:
# --- Literature comparison table ---
lit_rows = [
    {"reference": "Al-Waisy et al. 2025 [1]", "method": "Skin-DeepNet (HRNet+DBN+XGBoost fusion)",
     "dataset": "ISIC 2019 / HAM10000", "accuracy": "0.9965 / 1.000", "f1": "0.9954 / —"},
    {"reference": "Kaur et al. 2025 [2]", "method": "N-DCNN with hair removal + ACNN segmentation",
     "dataset": "ISIC 2020", "accuracy": "0.9340", "f1": "0.9398"},
    {"reference": "Naeem et al. 2024 [3]", "method": "SNC_Net Inception V3 + handcrafted entropy fusion",
     "dataset": "ISIC 2019", "accuracy": "0.9781", "f1": "0.9810"},
    {"reference": "Shahzaib et al. 2025 [25]", "method": "EHR + Deep Residual U-Net + DenseNet169",
     "dataset": "ISIC 2019", "accuracy": "0.9774", "f1": "—"},
    {"reference": "Albahli 2025 [37]", "method": "YOLOv8 multi-dataset",
     "dataset": "ISIC2020+HAM10000+PH2", "accuracy": "—", "f1": "0.905"},
    {"reference": "Bansal et al. 2022 [21]", "method": "Handcrafted + EfficientNet-B0 + ANN",
     "dataset": "HAM10000", "accuracy": "0.949", "f1": "—"},
]
# Append our rows
def _method_label(name):
    if name in ARCHES:
        return "single CNN (TL+focal+TTA)"
    if name == "ensemble":
        return "soft-vote ensemble"
    if name == "hybrid_fusion":
        return "handcrafted + ABCD + deep + SMOTE-Tomek + MLP/XGBoost"
    if name == "baseline_logistic":
        return "logistic regression on raw pixels"
    if name == "classical_ml_svm":
        return "HOG + Color + GLCM + PCA + RBF-SVM"
    return "n/a"

for _, r in df_pre.iterrows():
    lit_rows.append({"reference": "This work — " + r["method"],
                     "method": _method_label(r["method"]),
                     "dataset": "HAM10000 (binary, lesion-grouped)",
                     "accuracy": f'{r["accuracy"]:.4f}' if pd.notna(r["accuracy"]) else "—",
                     "f1":       f'{r["f1"]:.4f}'       if pd.notna(r["f1"])       else "—"})
lit_df = pd.DataFrame(lit_rows)
lit_df.to_csv(config.RESULTS_DIR / "literature_comparison.csv", index=False)
print(lit_df.to_string(index=False))

In [ ]:
# --- Write everything as Markdown into paper/tables_and_figures.md ---
from pathlib import Path

md_path = Path("/content/melanoma-detection-ham10000/paper/tables_and_figures.md")
local_md = config.PAPER_DIR / "tables_and_figures.md"

def df_to_md(df):
    return df.to_markdown(index=False)

def section(title, body):
    return f"## {title}\n\n{body}\n\n"

content = "# Tables and Figures (auto-generated by 10_aggregation.ipynb)\n\n"
content += section("Table I — Headline 9-method comparison (HAM10000 binary, test set)",
                   df_to_md(df_pre))
content += section("Table II — Ablations", df_to_md(ab_df))
content += section("Table III — Literature comparison", df_to_md(lit_df))
content += section("Figures (saved as PNG in MyDrive/melanoma/results/)",
                   "- `epoch_curves.png` — val F1 vs epoch overlay (6 CNNs)\n"
                   "- `roc_overlay.png` — 9 ROC curves on one axis\n"
                   "- `error_analysis.png` — 4 false positives + 4 false negatives from "
                   f"the best single CNN ({best_arch})\n"
                   "- per-arch `*_curves.png`, `*_confusion_matrix.png`, `*_roc_curve.png`, "
                   "`*_gradcam_grid.png`")

# Save in two places: Drive (for the user) and local repo (for git)
local_md.parent.mkdir(parents=True, exist_ok=True)
local_md.write_text(content, encoding="utf-8")
md_path.parent.mkdir(parents=True, exist_ok=True)
md_path.write_text(content, encoding="utf-8")
print("Wrote:", local_md)
print("Wrote:", md_path)

## Final summary

You should now have, in `MyDrive/melanoma/results/`:

- `comparison_table.csv` — the 9-row headline table for the IEEE paper.
- `ablation_table.csv` — 8 rows of ablation evidence (loss, TTA, threshold, ensemble).
- `literature_comparison.csv` — our six rows next to six prior-work numbers.
- `epoch_curves.png` — val F1 vs epoch overlay for the six CNNs.
- `roc_overlay.png` — nine ROC curves on one axis with AUC values in the legend.
- `error_analysis.png` — four false positives and four false negatives from
  the best single CNN, with the model's predicted probability shown.
- per-arch artefacts (curves, confusion matrices, Grad-CAM grids).

Plus, in `paper/tables_and_figures.md` (in this repo and on Drive), a
markdown rendering of all three tables ready to drop into Overleaf.